# Демо: классы и наследование

Прокликай Shift+Enter каждую ячейку и посмотри, как Python работает с классами: объявление, конструктор, `self`, атрибуты экземпляра и класса, наследование, `super()`, переопределение методов, MRO. В конце — три мини-задания.

## Часть 1. Объявление класса и создание объекта

Сейчас посмотрим, как `class` фиксирует шаблон, а вызов имени класса со скобками создаёт по этому шаблону объект. `__init__` запускается автоматически — туда передаются аргументы из вызова.

In [1]:
class Counter:
    def __init__(self, start=0):
        self.value = start

    def increment(self):
        self.value += 1

    def reset(self):
        self.value = 0

# Создание объекта — Python сам вызовет __init__
c = Counter(start=10)
print(type(c).__name__)   # Counter
print(c.value)            # 10

Counter
10


Каждый объект — независимый. У них свой набор атрибутов, методы у одного не задевают другого:

In [2]:
a = Counter()           # a.value = 0
b = Counter(start=5)    # b.value = 5

a.increment()
a.increment()
a.increment()

print(a.value, b.value)   # 3 5 — независимые счётчики
print(a is b)             # False — это два разных объекта

3 5
False


## Часть 2. Откуда берётся `self`

При вызове `c.increment()` Python под капотом превращает это в `Counter.increment(c)` — функцию, в которую первым аргументом передан сам объект. Этот объект попадает в параметр `self`. Покажем оба варианта в одном выводе:

In [3]:
c = Counter(start=100)

# Привычная форма
c.increment()
print(c.value)              # 101

# Эквивалент — явный вызов через имя класса
Counter.increment(c)
print(c.value)              # 102 — то же самое

101
102


Когда `self` нужен, а когда нет:

- `self.атрибут` — обращение к атрибуту объекта
- `self.метод()` — вызов другого метода того же объекта
- локальная переменная внутри метода — без `self`

In [4]:
class Calculator:
    def __init__(self):
        self.history = []

    def add(self, a, b):
        result = a + b              # локальная переменная — без self
        self.history.append(result) # атрибут объекта — с self
        return result

    def repeat_last(self, times):
        last = self.history[-1]      # снова атрибут
        return last * times

calc = Calculator()
calc.add(2, 3)
calc.add(10, 5)
print(calc.history)            # [5, 15]
print(calc.repeat_last(3))     # 45

[5, 15]
45


## Часть 3. Атрибут экземпляра vs атрибут класса

Атрибут, объявленный внутри `__init__` через `self.x = ...` — у каждого объекта свой. Атрибут, объявленный в теле `class` вне методов — общий для всех экземпляров (живёт на уровне класса).

In [5]:
class Employee:
    company = "Acme Corp"          # атрибут класса — общий

    def __init__(self, name):
        self.name = name             # атрибут экземпляра — свой у каждого

alice = Employee("Аня")
bob = Employee("Боря")

print(alice.name, bob.name)         # Аня Боря — свои у каждого
print(alice.company, bob.company)   # Acme Corp Acme Corp — общий

Аня Боря
Acme Corp Acme Corp


Если объекту присвоить атрибут с тем же именем — у него появится **свой** атрибут экземпляра, перекрывающий атрибут класса. У других объектов всё остаётся как было:

In [6]:
alice.company = "Beta Inc"        # создали атрибут экземпляра у alice

print(alice.company)                # Beta Inc — свой
print(bob.company)                  # Acme Corp — общий не задет
print(Employee.company)             # Acme Corp — атрибут класса не задет

Beta Inc
Acme Corp
Acme Corp


## Часть 4. Подводный камень: изменяемый атрибут класса

А что если атрибут класса — изменяемый объект (список, словарь)? Тогда `obj.attr.append(...)` модифицирует **общий** список, и изменение увидят все экземпляры. Знакомая ловушка из мира функций, теперь в мире классов:

In [7]:
class Project:
    tags = []   # изменяемый атрибут класса — ОБЩИЙ для всех экземпляров

    def __init__(self, name):
        self.name = name

p1 = Project("alpha")
p2 = Project("beta")

p1.tags.append("urgent")     # модифицируем общий список
print(p1.tags)                  # ['urgent']
print(p2.tags)                  # ['urgent']  — а должно быть [], это bug!
print(Project.tags)             # ['urgent']  — изменение на уровне класса

['urgent']
['urgent']
['urgent']


Правильное решение — инициализировать изменяемые атрибуты в `__init__`. Тогда у каждого экземпляра своя копия, ловушка не работает:

In [8]:
class ProjectFixed:
    def __init__(self, name):
        self.name = name
        self.tags = []   # каждый экземпляр получает свой пустой список

p1 = ProjectFixed("alpha")
p2 = ProjectFixed("beta")

p1.tags.append("urgent")
print(p1.tags)            # ['urgent']
print(p2.tags)            # []  — правильно, у каждого свой

['urgent']
[]


## Часть 5. Наследование: ребёнок берёт всё от родителя

`class Child(Parent):` создаёт класс, который **автоматически** получает все атрибуты и методы родителя. Можно добавить своё или переопределить уже существующее.

In [9]:
class User:
    def __init__(self, name, email):
        self.name = name
        self.email = email

    def greet(self):
        return f"Привет, я {self.name}"

class Admin(User):
    def ban(self, target_name):
        return f"{self.name} забанил {target_name}"

admin = Admin("Аня", "anya@acme.com")
print(admin.name)              # Аня — атрибут от User
print(admin.email)             # anya@acme.com — тоже от User
print(admin.greet())           # Привет, я Аня — метод от User
print(admin.ban("Spam Bot"))   # Аня забанил Spam Bot — свой

Аня
anya@acme.com
Привет, я Аня
Аня забанил Spam Bot


## Часть 6. Расширение `__init__` через `super()`

Дочернему классу часто нужны и родительские атрибуты, и свои. Вместо копирования логики из родителя зовут его `__init__` через `super()`. Можно представить как звонок родителю: «помоги проинициализировать своё, а я добавлю своё».

In [10]:
class User:
    def __init__(self, name, email):
        print(f"  User.__init__: name={name}, email={email}")
        self.name = name
        self.email = email

class Admin(User):
    def __init__(self, name, email, permissions):
        print(f"  Admin.__init__: вызывает super().__init__")
        super().__init__(name, email)        # родитель ставит name, email
        self.permissions = permissions       # ребёнок добавляет своё
        print(f"  Admin.__init__: добавил permissions={permissions}")

admin = Admin("Аня", "a@x.com", ["ban", "edit"])
print()
print("Все атрибуты:", admin.__dict__)

  Admin.__init__: вызывает super().__init__
  User.__init__: name=Аня, email=a@x.com
  Admin.__init__: добавил permissions=['ban', 'edit']

Все атрибуты: {'name': 'Аня', 'email': 'a@x.com', 'permissions': ['ban', 'edit']}


А что если забыть `super().__init__(...)` в наследнике? Атрибуты родителя просто не будут проставлены. Потом при попытке использовать их — `AttributeError`:

In [11]:
class BadAdmin(User):
    def __init__(self, name, email, permissions):
        # super().__init__(name, email)   <- забыли!
        self.permissions = permissions

bad = BadAdmin("Боря", "b@x.com", ["edit"])

try:
    print(bad.name)
except AttributeError as e:
    print(f"AttributeError: {e}")

AttributeError: 'BadAdmin' object has no attribute 'name'


## Часть 7. Переопределение методов и полиморфизм

Дочерний класс может написать свой метод с тем же именем — это **переопределение**. При вызове через объект дочернего класса сработает дочерняя версия. Один и тот же вызов `obj.sound()` ведёт себя по-разному в зависимости от типа объекта — это **полиморфизм**.

In [12]:
class Animal:
    def sound(self):
        return "..."

class Dog(Animal):
    def sound(self):
        return "гав"

class Cat(Animal):
    def sound(self):
        return "мяу"

for animal in [Animal(), Dog(), Cat()]:
    # Один вызов — три разных результата
    print(f"{type(animal).__name__}: {animal.sound()}")

Animal: ...
Dog: гав
Cat: мяу


Часто переопределённый метод хочет и родительскую логику, и свою. Тогда снова вызывают `super()`:

In [13]:
class User:
    def describe(self):
        return f"User({self.name})"

    def __init__(self, name):
        self.name = name

class Admin(User):
    def describe(self):
        base = super().describe()       # вызовет User.describe — "User(Аня)"
        return f"{base} [admin]"

admin = Admin("Аня")
print(admin.describe())   # User(Аня) [admin]

User(Аня) [admin]


## Часть 8. Многоуровневое наследование и MRO

Наследование может идти на несколько уровней: `C(B)`, `B(A)`. Тогда `C` получает всё от `B` плюс всё от `A`. Цепочка `super()` работает естественно.

In [14]:
class Animal:
    def __init__(self, name):
        self.name = name

class Pet(Animal):
    def __init__(self, name, owner):
        super().__init__(name)         # вызовет Animal.__init__
        self.owner = owner

class Dog(Pet):
    def __init__(self, name, owner, breed):
        super().__init__(name, owner)   # вызовет Pet.__init__ → Animal.__init__
        self.breed = breed

rex = Dog("Рекс", "Аня", "лабрадор")
print(rex.name, rex.owner, rex.breed)   # Рекс Аня лабрадор

Рекс Аня лабрадор


Порядок поиска атрибутов вверх по цепочке наследования называется **MRO** (Method Resolution Order). Можно посмотреть через `Class.__mro__`:

In [15]:
for cls in Dog.__mro__:
    print(cls.__name__)
# Dog
# Pet
# Animal
# object — общий предок всех классов в Python

Dog
Pet
Animal
object


При множественном наследовании (родителей несколько) Python проходит их слева направо — это и обеспечивает корректную работу `super()` даже в сложных иерархиях:

In [16]:
class A:
    def hi(self): return "A"

class B:
    def hi(self): return "B"

class C(A, B):     # A первый, B второй
    pass

print(C().hi())                 # A — Python взял первого родителя
print([c.__name__ for c in C.__mro__])   # ['C', 'A', 'B', 'object']

A
['C', 'A', 'B', 'object']


## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Напиши класс `BankAccount` с методами `deposit(amount)` и `withdraw(amount)`. Атрибут `balance` стартует с переданного в конструктор `initial`. Метод `withdraw` должен поднимать `ValueError`, если денег недостаточно.

**Задание 2.** Сделай класс `SavingsAccount`, который наследует от `BankAccount` и при создании принимает дополнительный параметр `interest_rate`. Добавь метод `apply_interest()`, который увеличивает баланс на `balance * interest_rate`.

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [2]:
# Задание 1
class BankAccount:
    def __init__(self, initial=0):
        self.balance = initial

    def deposit(self, amount):
        self.balance += amount

    def withdraw(self, amount):
        if self.balance < amount:
            raise ValueError('Недостаточно денег на балансе')
        else:
            self.balance -= amount

# Проверка:
acc = BankAccount(initial=100)
acc.deposit(50)
print(acc.balance)     # 150
acc.withdraw(80)
print(acc.balance)     # 70
acc.withdraw(1000)     # ValueError


150
70


ValueError: Недостаточно денег на балансе

In [5]:
# Задание 2
class SavingsAccount(BankAccount):
    def __init__(self, initial=0, interest_rate=0.01):
        super().__init__(initial)
        self.interest_rate = interest_rate

    def apply_interest(self):
        self.balance += self.balance * self.interest_rate

# Проверка:
sav = SavingsAccount(initial=1000, interest_rate=0.05)
sav.apply_interest()
print(sav.balance)     # 1050.0


1050.0


In [ ]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
class Box:
    contents = []   # атрибут класса

    def __init__(self, label):
        self.label = label

b1 = Box("red")
b2 = Box("blue")
b1.contents.append("apple")

print(b1.contents)   # apple
print(b2.contents)   # apple
print(Box.contents)  # apple


['apple']
['apple']
['apple']
